## Visualize OACNN TEST

In [1]:
def shift_point_cloud(pcd_gt, pcd_pred, points, offset):
    """Shifts pcd_pred along the X axis and adjusts its Z to maintain terrain coherence."""
    min_x = points[:, 0].min()
    max_x = points[:, 0].max()
    real_width = max_x - min_x

    # Compute height difference (Delta Z) between the ends of the X axis
    # Sample a small strip (5%) at the start and end to average the height
    start_mask = points[:, 0] < (min_x + real_width * 0.05)
    end_mask   = points[:, 0] > (max_x - real_width * 0.05)

    avg_z_start = points[start_mask, 2].mean()
    avg_z_end   = points[end_mask,   2].mean()

    # Compute slope (m = delta_z / delta_x)
    # How many metres the terrain rises/falls per metre along X
    slope = (avg_z_end - avg_z_start) / real_width

    # Compute Z offset from X offset
    # Moving the cloud 'offset' metres in X requires 'offset * slope' metres in Z
    z_offset = slope * offset

    # Apply the 3D translation (X and Z synchronized)
    pcd_pred.translate([offset, 0, z_offset])

### VISUALIZE GT AND PREDICTIONS SIDE BY SIDE

In [47]:
import laspy
import numpy as np
import open3d as o3d
import torch
import matplotlib.pyplot as plt

def visualize_side_by_side(path_las, pred_pth):

    try:
        preds = torch.load(pred_pth)
    except Exception as e:
        print(f"Error loading .pth file: {e}")
        return


    # Load the LAS file
    las = laspy.read(path_las)
    points = np.vstack((las.x, las.y, las.z)).T
    gt_labels = np.array(las.classification)

    center = points.mean(axis=0)
    points = points - center


    # Create the Open3D point cloud
    pcd_gt = o3d.geometry.PointCloud()
    pcd_gt.points = o3d.utility.Vector3dVector(points)

    # Colorize Ground Truth (IDs >= 3)
    max_label = int(gt_labels.max())
    color_palette = np.random.uniform(0, 1, size=(max_label + 1, 3))
    color_palette[:3] = [0.2, 0.2, 0.2]  # Background classes (0,1,2) in grey
    pcd_gt.colors = o3d.utility.Vector3dVector(color_palette[gt_labels])

    # Build the prediction cloud for the right side
    pcd_pred = o3d.geometry.PointCloud(pcd_gt)

    # Build label array
    num_points = points.shape[0]
    num_instances = preds['pred_masks'].shape[0]
    labels = np.full(num_points, -1, dtype=np.int32)

    # Assign unique ID to each mask
    for i in range(num_instances):
        mask = preds['pred_masks'][i].cpu().numpy().astype(bool)
        labels[mask] = i

    # Colorize predictions by assigned labels
    cmap = plt.get_cmap("tab20")
    colors = np.zeros((num_points, 3))

    # Background/uninstanced points (very dark grey)
    colors[labels == -1] = [0.15, 0.15, 0.15]

    # Detected trees
    for i in range(num_instances):
        colors[labels == i] = cmap(i % 20)[:3]
    pcd_pred.colors = o3d.utility.Vector3dVector(colors)


    shift_point_cloud(pcd_gt, pcd_pred, points, offset=25)  # Offset 25 m in X

    # Visualize
    print(f"Showing Side by Side: GT (Left) | Prediction (Right)")
    vis = o3d.visualization.Visualizer()
    vis.create_window(window_name="Instance Comparison - Santomera", width=1200, height=800)

    vis.add_geometry(pcd_gt)
    #vis.add_geometry(pcd_pred)


    opt = vis.get_render_option()
    opt.background_color = np.asarray([0.05, 0.05, 0.05])
    opt.point_size = 2.5

    vis.run()
    vis.destroy_window()


### VISUALIZE SEMANTIC ERROR MAP OF GT AND PREDICTIONS

In [3]:
import laspy
import numpy as np
import open3d as o3d
import torch

def visualize_error_map(path_las, pred_pth):
    # Load LAS and normalize
    las = laspy.read(path_las)
    points = np.vstack((las.x, las.y, las.z)).T

    # Normalize for optimal visualization
    center_xy = points[:, :2].mean(axis=0)
    z_min = points[:, 2].min()
    points[:, :2] -= center_xy
    points[:, 2] -= z_min

    # Load predictions and build binary mask
    preds = torch.load(pred_pth)
    num_points = points.shape[0]

    # Combine all instances into a single "Predicted Vegetation" mask
    is_tree_pred = np.zeros(num_points, dtype=bool)
    for i in range(preds['pred_masks'].shape[0]):
        mask = preds['pred_masks'][i].cpu().numpy().astype(bool)
        is_tree_pred |= mask  # OR operation to merge all tree masks

    # Build binary GT mask (classification >= 3)
    is_tree_gt = (las.classification >= 3)

    # Confusion-map colour logic
    colors = np.full((num_points, 3), 0.15)  # Default dark grey background

    # HIT (True Positive): Green
    # Both GT and prediction agree: tree
    hit_mask = is_tree_gt & is_tree_pred
    colors[hit_mask] = [0.0, 1.0, 0.0]  # Bright green

    # ERROR (False Positive or False Negative): Red
    # GT says tree but Pred does not, or vice versa
    error_mask = (is_tree_gt != is_tree_pred) & (is_tree_gt | is_tree_pred)
    colors[error_mask] = [1.0, 0.0, 0.0]  # Pure red

    # Create Open3D visualization
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    pcd.colors = o3d.utility.Vector3dVector(colors)

    # Use Visualizer to allow a custom background
    vis = o3d.visualization.Visualizer()
    vis.create_window(window_name="Error Analysis - Santomera", width=1200, height=800)

    # Add the point cloud
    vis.add_geometry(pcd)

    # Aesthetic settings: background and point size
    opt = vis.get_render_option()
    opt.background_color = np.asarray([0.05, 0.05, 0.05])  # Dark grey background
    opt.point_size = 2.5

    print("Error Map Generated:")
    print(f"Green: Hit (GT and Prediction agree)")
    print(f"Red: Error (Mismatch between GT and Prediction)")

    vis.run()
    vis.destroy_window()

### VISUALIZE 3D REPRESENTATION OF A SINGLE TREE

In [ ]:
import laspy
import numpy as np
import open3d as o3d
import torch

def visualize_single_tree_alpha(path_las, pred_pth, tree_index=0, alpha_val=0.2):
    try:
        preds = torch.load(pred_pth)
    except Exception as e:
        print(f"Error loading .pth file: {e}")
        return

    # Load LAS coordinates
    las = laspy.read(path_las)
    points_all = np.vstack((las.x, las.y, las.z)).T

    # Isolate the selected tree's points
    # Use the prediction mask
    mask = preds['pred_masks'][tree_index].cpu().numpy().astype(bool)
    tree_points = points_all[mask]

    if len(tree_points) < 4:
        print("The selected tree does not have enough points to build a 3D geometry.")
        return

    # CENTERING: Move the tree to the origin (0,0,0)
    # Prevents the Open3D camera from being placed far from the object
    tree_center = tree_points.mean(axis=0)
    tree_points = tree_points - tree_center

    # Create point cloud object
    tree_pcd = o3d.geometry.PointCloud()
    tree_pcd.points = o3d.utility.Vector3dVector(tree_points)
    tree_pcd.paint_uniform_color([0.5, 0.5, 0.5])  # Soft grey points

    # Generate 3D Alpha Shape geometry
    mesh_alpha = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(tree_pcd, alpha_val)
    mesh_alpha.compute_vertex_normals()

    # Mesh aesthetics
    mesh_alpha.paint_uniform_color([0.1, 0.8, 0.2])  # Forest green

    # Visualization
    print(f"Visualizing Tree ID:{tree_index} with Alpha:{alpha_val}")
    print(f"Detected height: {tree_points[:,2].max() - tree_points[:,2].min():.2f} m")

    vis = o3d.visualization.Visualizer()
    vis.create_window(window_name=f"Alpha Shape Geometry - Tree {tree_index}", width=1000, height=800)

    # Add the mesh (geometry) and the points (optional, for reference)
    vis.add_geometry(mesh_alpha)
    vis.add_geometry(tree_pcd)

    opt = vis.get_render_option()
    opt.background_color = np.asarray([0.05, 0.05, 0.05])
    opt.point_size = 3.0
    # Enable wireframe to see the triangle structure
    opt.mesh_show_wireframe = True

    vis.run()
    vis.destroy_window()


## Visualize all trees in 3D representation

In [ ]:
import laspy
import numpy as np
import open3d as o3d
import torch

def visualize_forest_alpha(path_las, pred_pth, alpha_val=0.15):
    try:
        preds = torch.load(pred_pth)
    except Exception as e:
        print(f"Error loading .pth file: {e}")
        return

    # Load LAS coordinates and colours
    las = laspy.read(path_las)
    points_all = np.vstack((las.x, las.y, las.z)).T

    # Extract RGB channels
    try:
        # Some LAS files use 16-bit (0-65535); normalize to 0-1
        red   = np.array(las.red)
        green = np.array(las.green)
        blue  = np.array(las.blue)

        # Detect 16-bit vs 8-bit
        max_val = 65535 if np.max(red) > 255 else 255
        rgb_all = np.vstack((red, green, blue)).T / max_val
    except AttributeError:
        print("Warning: LAS file contains no colour information (RGB).")
        return

    # Centre the scene for Open3D
    scene_center = points_all.mean(axis=0)
    points_all_centered = points_all - scene_center

    num_instances = preds['pred_masks'].shape[0]
    geometries = []

    print(f"Computing real colours for {num_instances} instances...")

    # Loop over each tree
    for i in range(num_instances):
        mask = preds['pred_masks'][i].cpu().numpy().astype(bool)

        tree_points = points_all_centered[mask]
        tree_colors = rgb_all[mask]

        if len(tree_points) < 10:
            continue

        # --- AVERAGE COLOUR COMPUTATION ---
        # Average R, G, B across all points of this tree
        avg_color = np.mean(tree_colors, axis=0)

        # Generate Alpha Shape mesh
        pcd_temp = o3d.geometry.PointCloud()
        pcd_temp.points = o3d.utility.Vector3dVector(tree_points)

        try:
            mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(pcd_temp, alpha_val)
            mesh.compute_vertex_normals()

            # Apply the tree's real average colour
            mesh.paint_uniform_color(avg_color)
            geometries.append(mesh)
        except:
            continue

    # Visualization
    print(f"Showing {len(geometries)} trees with their real colours.")
    vis = o3d.visualization.Visualizer()
    vis.create_window(window_name="Santomera: Digital Twin (Alpha Shapes RGB)", width=1200, height=800)

    for g in geometries:
        vis.add_geometry(g)

    opt = vis.get_render_option()
    opt.background_color = np.asarray([0.05, 0.05, 0.05])
    opt.mesh_show_wireframe = False  # Disabled for a more realistic look
    opt.light_on = True              # Enable lighting to give volume to the meshes

    vis.run()
    vis.destroy_window()

In [16]:
import os

# Execution
base_path_preds = "../../data/Santomera/result/OACNN_0"
base_path_gt = "../../data/Santomera/tiles/trees/labeled"

for file in os.listdir(base_path_preds):
    if file.endswith(".pth"):
        pred_path = os.path.join(base_path_preds, file)
        las_name = file.replace("_pred.pth", ".las")
        las_path = os.path.join(base_path_gt, las_name)
        print(f"Visualizing: {las_name} with prediction {file}")
        visualize_side_by_side(las_path, pred_path)

Visualizando: tile_2260.las con predicción tile_2260_pred.pth
Mostrando Lado a Lado: GT (Izquierda) | Predicción (Derecha)
Visualizando: tile_2303.las con predicción tile_2303_pred.pth
Mostrando Lado a Lado: GT (Izquierda) | Predicción (Derecha)
Visualizando: tile_2438.las con predicción tile_2438_pred.pth
Mostrando Lado a Lado: GT (Izquierda) | Predicción (Derecha)
Visualizando: tile_3330.las con predicción tile_3330_pred.pth
Mostrando Lado a Lado: GT (Izquierda) | Predicción (Derecha)
Visualizando: tile_3745.las con predicción tile_3745_pred.pth
Mostrando Lado a Lado: GT (Izquierda) | Predicción (Derecha)
Visualizando: tile_3747.las con predicción tile_3747_pred.pth
Mostrando Lado a Lado: GT (Izquierda) | Predicción (Derecha)
Visualizando: tile_3770.las con predicción tile_3770_pred.pth
Mostrando Lado a Lado: GT (Izquierda) | Predicción (Derecha)
Visualizando: tile_3771.las con predicción tile_3771_pred.pth
Mostrando Lado a Lado: GT (Izquierda) | Predicción (Derecha)
Visualizando: ti

In [17]:
import os

# Execution
base_path_preds = "../../data/Santomera/result/OACNN"
base_path_gt = "../../data/Santomera/tiles/trees/labeled"

for file in os.listdir(base_path_preds):
    if file.endswith(".pth"):
        pred_path = os.path.join(base_path_preds, file)
        las_name = file.replace("_pred.pth", ".las")
        las_path = os.path.join(base_path_gt, las_name)
        print(f"Visualizing: {las_name} with prediction {file}")
        visualize_error_map(las_path, pred_path)

Visualizando: tile_2303.las con predicción tile_2303_pred.pth
Mapa de Errores Generado:
Verde: Acierto (Coinciden GT y Pred)
Rojo: Error (Inconsistencia entre GT y Pred)
Visualizando: tile_3745.las con predicción tile_3745_pred.pth
Mapa de Errores Generado:
Verde: Acierto (Coinciden GT y Pred)
Rojo: Error (Inconsistencia entre GT y Pred)
Visualizando: tile_3770.las con predicción tile_3770_pred.pth
Mapa de Errores Generado:
Verde: Acierto (Coinciden GT y Pred)
Rojo: Error (Inconsistencia entre GT y Pred)
Visualizando: tile_3771.las con predicción tile_3771_pred.pth
Mapa de Errores Generado:
Verde: Acierto (Coinciden GT y Pred)
Rojo: Error (Inconsistencia entre GT y Pred)
Visualizando: tile_3859.las con predicción tile_3859_pred.pth
Mapa de Errores Generado:
Verde: Acierto (Coinciden GT y Pred)
Rojo: Error (Inconsistencia entre GT y Pred)
Visualizando: tile_98.las con predicción tile_98_pred.pth
Mapa de Errores Generado:
Verde: Acierto (Coinciden GT y Pred)
Rojo: Error (Inconsistencia e

In [16]:
import os

# Execution
base_path_preds = "../../data/Santomera/result/OACNN"
base_path_gt = "../../data/Santomera/tiles/trees/labeled"

for file in os.listdir(base_path_preds):
    if file.endswith(".pth"):
        pred_path = os.path.join(base_path_preds, file)
        las_name = file.replace("_pred.pth", ".las")
        las_path = os.path.join(base_path_gt, las_name)
        print(f"Visualizing: {las_name} with prediction {file}")
        visualize_single_tree_alpha(las_path, pred_path, tree_index=2, alpha_val=0.5)

Visualizando: tile_2303.las con predicción tile_2303_pred.pth
Visualizando Árbol ID:2 con Alpha:0.5
Altura detectada: 7.25 m
Visualizando: tile_3745.las con predicción tile_3745_pred.pth
Visualizando Árbol ID:2 con Alpha:0.5
Altura detectada: 8.35 m
Visualizando: tile_3770.las con predicción tile_3770_pred.pth
Visualizando Árbol ID:2 con Alpha:0.5
Altura detectada: 3.63 m
Visualizando: tile_3771.las con predicción tile_3771_pred.pth
Visualizando Árbol ID:2 con Alpha:0.5
Altura detectada: 6.31 m
Visualizando: tile_3859.las con predicción tile_3859_pred.pth
Visualizando Árbol ID:2 con Alpha:0.5
Altura detectada: 13.31 m
Visualizando: tile_98.las con predicción tile_98_pred.pth
Visualizando Árbol ID:2 con Alpha:0.5
Altura detectada: 5.33 m


In [ ]:
import os

# Execution
base_path_preds = "../../data/Santomera/result/OACNN"
base_path_gt = "../../data/Santomera/tiles/trees/labeled"

for file in os.listdir(base_path_preds):
    if file.endswith(".pth"):
        pred_path = os.path.join(base_path_preds, file)
        las_name = file.replace("_pred.pth", ".las")
        las_path = os.path.join(base_path_gt, las_name)
        print(f"Visualizing: {las_name} with prediction {file}")
        visualize_forest_alpha(las_path, pred_path, alpha_val=0.4)

Visualizando: tile_2260.las con predicción tile_2260_pred.pth
Calculando colores reales para 19 instancias...
Mostrando 19 árboles con sus colores reales.
Visualizando: tile_2303.las con predicción tile_2303_pred.pth
Calculando colores reales para 30 instancias...


KeyboardInterrupt: 

## Conference visualization

In [50]:
import os

# Execution
base_path_preds = "../../data/Santomera/result/OACNN_100"
base_path_gt = "../../data/Santomera/tiles/trees/labeled"
tile = "tile_3330"

pred_path = os.path.join(base_path_preds, tile + "_pred.pth")

las_path = os.path.join(base_path_gt, tile + ".las")
print(f"Visualizing: {tile} with prediction {tile}_pred.pth")
visualize_side_by_side(las_path, pred_path)

Visualizando: tile_3330 con predicción tile_3330_pred.pth
Mostrando Lado a Lado: GT (Izquierda) | Predicción (Derecha)
